# UC_AutoMaiL α版を試す

配布wheelをPython 3.12以降のCPU環境で使い、分類、線形カルマン、粒子フィルタ、GCNの操作と出力を確認します。

- **Colab**。このノートをアップロードして最初のセルを実行し、求められたら `uc_automail-0.4.1a1-py3-none-any.whl` を選びます。
- **ローカルJupyter**。リポジトリのルートまたは `PyPI/` から実行すると、配布候補のwheelを自動で選びます。別の場所にある場合は、下の `wheel` にパスを指定します。

標準の試用は「分類」と「1系列の時系列」です。「詳細分析」のセルは必要に応じて実行できます。合成データは操作を確認するためのサンプルです。

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone
from time import perf_counter

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12 or later runtime.")

version = "0.4.1a1"
filename = f"uc_automail-{version}-py3-none-any.whl"
bundled = Path("candidates") / version / filename
local_paths = (Path(filename), bundled, Path("PyPI") / bundled)
wheel_override = os.environ.get("DIY_AUTOML_TRIAL_WHEEL")
if wheel_override:
    wheel = Path(wheel_override)
else:
    wheel = next((path for path in local_paths if path.is_file()), Path(filename))
if not wheel.is_file():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Set wheel to the path of {filename}.") from None
    uploaded = files.upload()
    if filename not in uploaded:
        raise ValueError(f"Select {filename}.")
    wheel = Path.cwd() / filename
    wheel.write_bytes(uploaded[filename])
if wheel.name != filename:
    raise ValueError(f"Select {filename}.")
wheel = wheel.resolve()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
work = Path.cwd() / "runs" / f"uc-automail-alpha-{stamp}"
work.mkdir(parents=True)
environment = work / "venv"
subprocess.run([sys.executable, "-m", "venv", str(environment)], check=True)
python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

def run_python(*args):
    process = subprocess.run(
        [str(python), "-I", "-X", "utf8", *map(str, args)],
        cwd=work, env={**os.environ, "MPLBACKEND": "Agg"},
        capture_output=True, encoding="utf-8", errors="replace",
    )
    if process.stdout:
        print(process.stdout, end="")
    if process.returncode:
        print(process.stderr, file=sys.stderr, end="")
    process.check_returncode()

def trial(command, *args, output):
    start = perf_counter()
    run_python("-m", "uc_automail", command, *args, "--output", output)
    run_python(
        "-c",
        "from diy_automl.validation import validate; import sys; validate(sys.argv[1])",
        output,
    )
    print(f"{command}: {perf_counter() - start:.1f} s including validation")

wheelhouse = os.environ.get("DIY_AUTOML_TRIAL_WHEELHOUSE")
install_options = ["--no-index", "--find-links", str(Path(wheelhouse).resolve())] if wheelhouse else []
start = perf_counter()
run_python("-m", "pip", "--disable-pip-version-check", "install", "--quiet", *install_options, wheel)
run_python("-m", "pip", "check")
run_python("-c", "import uc_automail as uc_al; print(uc_al.__version__)")
run_python("-m", "uc_automail", "--version")
print(f"Install: {perf_counter() - start:.1f} s")
print(f"Workspace: {work}")

## 分類を試す

scikit-learnのIris（150件）をCSVに保存し、教師あり5候補を比較します。点検用Excel、選定の進捗ログ、詳細図付きの評価Excelを作ります。

In [ ]:
run_python("-c", """
from sklearn.datasets import load_iris
load_iris(as_frame=True).frame.to_csv("iris.csv", index=False)
""")
trial("audit-data", "--csv", "iris.csv", "--target", "target", output="iris-audit")
trial("select", "--csv", "iris.csv", "--target", "target",
      "--progress", "excel", output="iris-selection")
run_python("-m", "uc_automail", "report", "--run", "iris-selection",
           "--charts", "detail", "--output", "iris-report")
run_python("-c", """
import pandas as pd
from diy_automl.excel_report_validation import validate_excel_report

report = validate_excel_report("iris-report")
print(f"Detailed Excel: {report['figure_count']} figures")
book = "iris-report/report_with_figures.xlsx"
for sheet in ("SL_CV_Mean", "SL_Test", "SL_Test_Intervals"):
    print()
    print(sheet)
    print(pd.read_excel(book, sheet_name=sheet).to_string(index=False))
""")

## 1系列の時系列を試す

位置がなめらかに変化する合成系列で、最初の24時点を使って1ステップ先のカルマン予測と直近値を比較します。残りは選定後の評価に使います。

In [ ]:
run_python("-c", """
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)
step = np.arange(80)
truth = 0.3*step + 0.025*step**2
observed = truth + rng.normal(0, 0.4, size=step.size)
pd.DataFrame({"step": step, "observed": observed, "truth": truth}).to_csv("track.csv", index=False)
""")
trial(
    "kalman-track", "--csv", "track.csv", "--time-column", "step",
    "--observation-column", "observed", "--reference-column", "truth",
    "--observation-sd", "0.4", "--process-noise", "0.0025",
    "--pilot-rows", "24", "--min-rmse-gain", "0.05", "--compact-report",
    output="kalman",
)

## 詳細分析を試す

同じ環境でEKF・UKF・粒子フィルタを理論モデル上で比較し、ミニグラフでGCNとロジスティック回帰の分類を比較します。グラフのサンプルは辺の与え方と出力を確認する目的で作っています。

In [ ]:
trial("particle-benchmark", "--steps", "30", "--trajectories", "8", "--particles", "128", output="filters")
run_python("-c", """
from pathlib import Path
import numpy as np
rng = np.random.default_rng(42)
nodes, edges = [], []
for label, center in (("A", (2.0, 1.0)), ("B", (1.0, 2.0))):
    features = rng.normal(center, 0.45, size=(12, 2))
    for i, (first, second) in enumerate(features):
        nodes.append(f"{label}{i}\\t{first:.6f}\\t{second:.6f}\\t{label}\\n")
        edges.append(f"{label}{i}\\t{label}{(i + 1) % len(features)}\\n")
Path("nodes.tsv").write_text("".join(nodes), encoding="utf-8")
Path("edges.tsv").write_text("".join(edges), encoding="utf-8")
""")
trial(
    "gnn-compare", "--nodes", "nodes.tsv", "--edges", "edges.tsv",
    "--train-per-class", "4", "--epochs", "40", output="graph",
)

## 結果を手元で開く

保存したExcel、配列、manifestと入力をZIPにまとめます。Colabでは最後に表示されたパスのZIPをファイル欄からダウンロードできます。

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive = work / "uc-automail-trial-results.zip"
inputs = ("iris.csv", "track.csv", "nodes.tsv", "edges.tsv")
runs = ("iris-audit", "iris-selection", "iris-report", "kalman", "filters", "graph")
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as results:
    for name in inputs:
        if (work / name).is_file():
            results.write(work / name, name)
    for name in runs:
        for path in (work / name).rglob("*"):
            if path.is_file():
                results.write(path, path.relative_to(work))
print(archive)